In [1]:

# =========================================================
# CELL 1: ĐỌC DỮ LIỆU VÀ CHIA TRAIN / VALIDATION / TEST
# NOTEBOOK: 02_Preprocessing.ipynb
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Xác định thư mục dự án
PROJECT_DIR = Path(
    r"C:\CreditAI_Project_Starter\CreditAI_Project"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Đọc lại dữ liệu gốc
DATA_PATH = RAW_DIR / "default of credit card clients.xls"

df = pd.read_excel(DATA_PATH, header=1)

target_col = "default payment next month"

# 3. Tách đặc trưng và nhãn mục tiêu
X = df.drop(columns=[target_col, "ID"])
y = df[target_col]

# 4. Chia Train 70% và phần còn lại 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# 5. Chia 30% còn lại thành Validation và Test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

# 6. Kiểm tra kích thước và tỷ lệ default
def show_split_info(name, X_part, y_part):
    print(
        f"{name}: "
        f"{X_part.shape[0]} dòng, "
        f"{X_part.shape[1]} đặc trưng, "
        f"Default = {y_part.mean() * 100:.2f}%"
    )

print("===== KẾT QUẢ CHIA DỮ LIỆU =====")

show_split_info("Train", X_train, y_train)
show_split_info("Validation", X_valid, y_valid)
show_split_info("Test", X_test, y_test)

# 7. Kiểm tra không trùng khách hàng giữa các tập
assert set(X_train.index).isdisjoint(X_valid.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_valid.index).isdisjoint(X_test.index)

assert len(X_train) + len(X_valid) + len(X_test) == len(df)

print("\nKhông có bản ghi trùng giữa các tập.")

# 8. Lưu dữ liệu đã chia
for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test)
]:
    part = X_part.copy()
    part[target_col] = y_part
    part.to_csv(
        PROCESSED_DIR / f"{name}_raw_split.csv",
        index=True,
        index_label="source_index",
        encoding="utf-8-sig"
    )

print("\nĐã lưu 3 tập dữ liệu vào data/processed/")


===== KẾT QUẢ CHIA DỮ LIỆU =====
Train: 21000 dòng, 23 đặc trưng, Default = 22.12%
Validation: 4500 dòng, 23 đặc trưng, Default = 22.11%
Test: 4500 dòng, 23 đặc trưng, Default = 22.13%

Không có bản ghi trùng giữa các tập.

Đã lưu 3 tập dữ liệu vào data/processed/


In [2]:

# =========================================================
# CELL 2: XÁC ĐỊNH BỘ ĐẶC TRƯNG CHO MÔ HÌNH
# =========================================================

# 1. Các biến tài chính và lịch sử thanh toán
financial_features = (
    ["LIMIT_BAL"]
    + ["PAY_0", "PAY_2", "PAY_3",
       "PAY_4", "PAY_5", "PAY_6"]
    + [f"BILL_AMT{i}" for i in range(1, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 7)]
)

# 2. Các biến nhân khẩu học để đánh giá công bằng
demographic_features = [
    "SEX", "EDUCATION", "MARRIAGE", "AGE"
]

# 3. Kiểm tra không bỏ sót hoặc trùng đặc trưng
assert len(financial_features) == 19
assert len(demographic_features) == 4
assert set(financial_features).isdisjoint(demographic_features)

assert set(financial_features + demographic_features) == set(X.columns)

# 4. Tạo dữ liệu đầu vào mô hình
X_train_model = X_train[financial_features].copy()
X_valid_model = X_valid[financial_features].copy()
X_test_model = X_test[financial_features].copy()

# 5. Lưu thông tin nhân khẩu học riêng để đánh giá
fairness_train = X_train[demographic_features].copy()
fairness_valid = X_valid[demographic_features].copy()
fairness_test = X_test[demographic_features].copy()

# 6. Kiểm tra kích thước
print("===== BỘ ĐẶC TRƯNG MÔ HÌNH =====")
print("Số đặc trưng tài chính:", len(financial_features))
print("Số biến nhân khẩu học:", len(demographic_features))

print("\n===== KÍCH THƯỚC DỮ LIỆU =====")
print("Train:", X_train_model.shape)
print("Validation:", X_valid_model.shape)
print("Test:", X_test_model.shape)

print("\n===== DANH SÁCH ĐẶC TRƯNG MÔ HÌNH =====")
for i, feature in enumerate(financial_features, start=1):
    print(f"{i:02d}. {feature}")

# 7. Kiểm tra dữ liệu gốc chưa bị thay đổi
assert X_train_model.shape == (21000, 19)
assert X_valid_model.shape == (4500, 19)
assert X_test_model.shape == (4500, 19)

print("\nCELL 2 HOÀN THÀNH!")


===== BỘ ĐẶC TRƯNG MÔ HÌNH =====
Số đặc trưng tài chính: 19
Số biến nhân khẩu học: 4

===== KÍCH THƯỚC DỮ LIỆU =====
Train: (21000, 19)
Validation: (4500, 19)
Test: (4500, 19)

===== DANH SÁCH ĐẶC TRƯNG MÔ HÌNH =====
01. LIMIT_BAL
02. PAY_0
03. PAY_2
04. PAY_3
05. PAY_4
06. PAY_5
07. PAY_6
08. BILL_AMT1
09. BILL_AMT2
10. BILL_AMT3
11. BILL_AMT4
12. BILL_AMT5
13. BILL_AMT6
14. PAY_AMT1
15. PAY_AMT2
16. PAY_AMT3
17. PAY_AMT4
18. PAY_AMT5
19. PAY_AMT6

CELL 2 HOÀN THÀNH!


In [3]:

# =========================================================
# CELL 3: FEATURE ENGINEERING - XÂY DỰNG ĐẶC TRƯNG MỚI
# =========================================================

status_cols = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

# 1. Hàm xây dựng đặc trưng
def create_features(data):
    result = data.copy()

    # Nhóm lịch sử thanh toán
    result["DELAY_COUNT"] = (
        result[status_cols] > 0
    ).sum(axis=1)

    result["SEVERE_DELAY_COUNT"] = (
        result[status_cols] >= 2
    ).sum(axis=1)

    result["MAX_DELAY"] = (
        result[status_cols].max(axis=1)
    )

    # Nhóm số tiền thanh toán
    result["ZERO_PAYMENT_COUNT"] = (
        result[payment_cols] == 0
    ).sum(axis=1)

    result["TOTAL_PAYMENT_6M"] = (
        result[payment_cols].sum(axis=1)
    )

    result["AVG_PAYMENT_6M"] = (
        result[payment_cols].mean(axis=1)
    )

    # Nhóm số tiền sao kê
    result["AVG_BILL_6M"] = (
        result[bill_cols].mean(axis=1)
    )

    result["BILL_CHANGE"] = (
        result["BILL_AMT1"] - result["BILL_AMT6"]
    )

    # Nhóm tỷ lệ sao kê / hạn mức
    # LIMIT_BAL > 0 đã được kiểm tra ở EDA
    result["UTILIZATION_RATIO"] = (
        result["BILL_AMT1"] / result["LIMIT_BAL"]
    )

    result["AVG_UTILIZATION_6M"] = (
        result["AVG_BILL_6M"] / result["LIMIT_BAL"]
    )

    return result


# 2. Áp dụng riêng cho từng tập dữ liệu
X_train_fe = create_features(X_train_model)
X_valid_fe = create_features(X_valid_model)
X_test_fe = create_features(X_test_model)

# 3. Kiểm tra số đặc trưng
new_features = [
    col for col in X_train_fe.columns
    if col not in financial_features
]

print("===== KẾT QUẢ FEATURE ENGINEERING =====")

print("Số đặc trưng gốc:", len(financial_features))
print("Số đặc trưng mới:", len(new_features))
print("Tổng số đặc trưng:", X_train_fe.shape[1])

print("\n===== KÍCH THƯỚC DỮ LIỆU =====")
print("Train:", X_train_fe.shape)
print("Validation:", X_valid_fe.shape)
print("Test:", X_test_fe.shape)

print("\n===== DANH SÁCH ĐẶC TRƯNG MỚI =====")
for i, col in enumerate(new_features, start=1):
    print(f"{i:02d}. {col}")

print("\n===== XEM THỬ DỮ LIỆU =====")
display(X_train_fe[new_features].head())

# 4. Kiểm tra tính hợp lệ
assert X_train_fe.shape[1] == 29
assert X_valid_fe.shape[1] == 29
assert X_test_fe.shape[1] == 29

assert X_train_fe.columns.equals(X_valid_fe.columns)
assert X_train_fe.columns.equals(X_test_fe.columns)

assert np.isfinite(
    X_train_fe.select_dtypes(include="number").to_numpy()
).all()

assert np.isfinite(
    X_valid_fe.select_dtypes(include="number").to_numpy()
).all()

assert np.isfinite(
    X_test_fe.select_dtypes(include="number").to_numpy()
).all()

print("\nCELL 3 HOÀN THÀNH!")


===== KẾT QUẢ FEATURE ENGINEERING =====
Số đặc trưng gốc: 19
Số đặc trưng mới: 10
Tổng số đặc trưng: 29

===== KÍCH THƯỚC DỮ LIỆU =====
Train: (21000, 29)
Validation: (4500, 29)
Test: (4500, 29)

===== DANH SÁCH ĐẶC TRƯNG MỚI =====
01. DELAY_COUNT
02. SEVERE_DELAY_COUNT
03. MAX_DELAY
04. ZERO_PAYMENT_COUNT
05. TOTAL_PAYMENT_6M
06. AVG_PAYMENT_6M
07. AVG_BILL_6M
08. BILL_CHANGE
09. UTILIZATION_RATIO
10. AVG_UTILIZATION_6M

===== XEM THỬ DỮ LIỆU =====


,DELAY_COUNT,SEVERE_DELAY_COUNT,MAX_DELAY,ZERO_PAYMENT_COUNT,TOTAL_PAYMENT_6M,AVG_PAYMENT_6M,AVG_BILL_6M,BILL_CHANGE,UTILIZATION_RATIO,AVG_UTILIZATION_6M
11018,0,0,0,0,35100,5850.000000,155590.666667,280,0.975612,0.972442
1710,1,1,2,1,29782,4963.666667,43046.500000,-24205,0.283830,0.430465
4618,0,0,0,0,12156,2026.000000,57302.000000,30918,0.921525,0.716275
5482,0,0,0,4,1786,297.666667,1704.333333,7286,0.364300,0.085217
26187,0,0,-2,5,2070,345.000000,345.000000,0,0.000000,0.006900



CELL 3 HOÀN THÀNH!


In [4]:

# =========================================================
# CELL 4: KIỂM TRA CHẤT LƯỢNG SAU FEATURE ENGINEERING
# =========================================================

# 1. Kiểm tra số giá trị thiếu và vô cực
print("===== KIỂM TRA NaN VÀ INFINITY =====")

for name, data in [
    ("Train", X_train_fe),
    ("Validation", X_valid_fe),
    ("Test", X_test_fe)
]:
    nan_count = int(data.isna().sum().sum())
    inf_count = int(np.isinf(data.to_numpy()).sum())

    print(
        f"{name}: NaN = {nan_count}, "
        f"Infinity = {inf_count}"
    )

# 2. Thống kê 10 đặc trưng mới trên Train
print("\n===== THỐNG KÊ ĐẶC TRƯNG MỚI - TRAIN =====")

feature_stats = X_train_fe[new_features].describe().T

display(feature_stats.round(3))

# 3. Kiểm tra phạm vi các biến đếm
print("\n===== KIỂM TRA BIẾN ĐẾM =====")

count_features = [
    "DELAY_COUNT",
    "SEVERE_DELAY_COUNT",
    "ZERO_PAYMENT_COUNT"
]

for col in count_features:
    min_value = X_train_fe[col].min()
    max_value = X_train_fe[col].max()

    print(f"{col}: min = {min_value}, max = {max_value}")

    assert min_value >= 0
    assert max_value <= 6

# 4. Kiểm tra biến dư thừa
print("\n===== KIỂM TRA BIẾN DƯ THỪA =====")

for name, data in [
    ("Train", X_train_fe),
    ("Validation", X_valid_fe),
    ("Test", X_test_fe)
]:
    is_equal = np.allclose(
        data["TOTAL_PAYMENT_6M"],
        data["AVG_PAYMENT_6M"] * 6
    )

    print(f"{name}: TOTAL = AVG x 6 -> {is_equal}")
    assert is_equal

# 5. Loại AVG_PAYMENT_6M khỏi bộ đặc trưng
# Giữ nguyên X_train_fe để có thể thử nghiệm sau
redundant_features = ["AVG_PAYMENT_6M"]

X_train_ready = X_train_fe.drop(
    columns=redundant_features
).copy()

X_valid_ready = X_valid_fe.drop(
    columns=redundant_features
).copy()

X_test_ready = X_test_fe.drop(
    columns=redundant_features
).copy()

# 6. Kiểm tra kết quả
print("\n===== BỘ ĐẶC TRƯNG SAU KIỂM TRA =====")

print("Train:", X_train_ready.shape)
print("Validation:", X_valid_ready.shape)
print("Test:", X_test_ready.shape)

assert X_train_ready.shape[1] == 28
assert X_valid_ready.shape[1] == 28
assert X_test_ready.shape[1] == 28

assert X_train_ready.columns.equals(X_valid_ready.columns)
assert X_train_ready.columns.equals(X_test_ready.columns)

# 7. Lưu bảng thống kê
feature_stats.to_csv(
    PROCESSED_DIR / "feature_engineering_statistics.csv",
    encoding="utf-8-sig"
)

print("\nCELL 4 HOÀN THÀNH!")


===== KIỂM TRA NaN VÀ INFINITY =====
Train: NaN = 0, Infinity = 0
Validation: NaN = 0, Infinity = 0
Test: NaN = 0, Infinity = 0

===== THỐNG KÊ ĐẶC TRƯNG MỚI - TRAIN =====


,count,mean,std,min,25%,50%,75%,max
DELAY_COUNT,21000.0,0.839,1.556,0.000,0.000,0.000,1.000,6.000
SEVERE_DELAY_COUNT,21000.0,0.713,1.465,0.000,0.000,0.000,1.000,6.000
MAX_DELAY,21000.0,0.439,1.340,-2.000,0.000,0.000,2.000,8.000
ZERO_PAYMENT_COUNT,21000.0,1.228,1.714,0.000,0.000,0.000,2.000,6.000
TOTAL_PAYMENT_6M,21000.0,31749.356,60451.116,0.000,6695.000,14500.000,34000.000,3764066.000
AVG_PAYMENT_6M,21000.0,5291.559,10075.186,0.000,1115.833,2416.667,5666.667,627344.333
AVG_BILL_6M,21000.0,44966.054,63332.429,-20913.333,4856.208,20982.083,57083.125,877313.833
BILL_CHANGE,21000.0,12356.541,43421.261,-399983.000,-3000.250,909.500,19930.750,518602.000
UTILIZATION_RATIO,21000.0,0.423,0.410,-0.620,0.022,0.316,0.829,6.455
AVG_UTILIZATION_6M,21000.0,0.373,0.351,-0.233,0.031,0.285,0.687,5.364



===== KIỂM TRA BIẾN ĐẾM =====
DELAY_COUNT: min = 0, max = 6
SEVERE_DELAY_COUNT: min = 0, max = 6
ZERO_PAYMENT_COUNT: min = 0, max = 6

===== KIỂM TRA BIẾN DƯ THỪA =====
Train: TOTAL = AVG x 6 -> True
Validation: TOTAL = AVG x 6 -> True
Test: TOTAL = AVG x 6 -> True

===== BỘ ĐẶC TRƯNG SAU KIỂM TRA =====
Train: (21000, 28)
Validation: (4500, 28)
Test: (4500, 28)

CELL 4 HOÀN THÀNH!


In [5]:

# =========================================================
# CELL 5: LƯU DỮ LIỆU SAU FEATURE ENGINEERING
# =========================================================

import json

# 1. Định nghĩa các bộ dữ liệu cần lưu
datasets = {
    "train": (X_train_ready, y_train),
    "valid": (X_valid_ready, y_valid),
    "test": (X_test_ready, y_test)
}

# 2. Lưu từng tập với nhãn mục tiêu
for name, (X_part, y_part) in datasets.items():
    output = X_part.copy()
    output[target_col] = y_part

    file_path = PROCESSED_DIR / f"{name}_features.csv"

    output.to_csv(
        file_path,
        index=True,
        index_label="source_index",
        encoding="utf-8-sig"
    )

    print(f"Đã lưu {name}: {output.shape} -> {file_path.name}")

# 3. Lưu danh sách đặc trưng
feature_config = {
    "target": target_col,
    "base_features": financial_features,
    "engineered_features": [
        col for col in X_train_ready.columns
        if col not in financial_features
    ],
    "model_features": X_train_ready.columns.tolist(),
    "excluded_demographics": demographic_features,
    "random_state": 42
}

config_path = PROCESSED_DIR / "feature_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        feature_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nĐã lưu cấu hình:", config_path.name)

# 4. Lưu dữ liệu nhân khẩu học riêng
fairness_datasets = {
    "train": (fairness_train, y_train),
    "valid": (fairness_valid, y_valid),
    "test": (fairness_test, y_test)
}

for name, (X_demo, y_part) in fairness_datasets.items():
    output = X_demo.copy()
    output[target_col] = y_part

    file_path = PROCESSED_DIR / f"{name}_fairness.csv"

    output.to_csv(
        file_path,
        index=True,
        index_label="source_index",
        encoding="utf-8-sig"
    )

print("Đã lưu 3 tập dữ liệu phục vụ đánh giá công bằng.")

# 5. Đọc thử file Train để xác nhận
check_train = pd.read_csv(
    PROCESSED_DIR / "train_features.csv",
    index_col="source_index"
)

print("\n===== KIỂM TRA FILE ĐÃ LƯU =====")
print("Kích thước Train:", check_train.shape)
print("Số giá trị thiếu:", check_train.isna().sum().sum())
print("Tỷ lệ default:", round(
    check_train[target_col].mean() * 100, 2
), "%")

assert check_train.shape == (21000, 29)
assert check_train.columns.tolist() == (
    X_train_ready.columns.tolist() + [target_col]
)

print("\nCELL 5 HOÀN THÀNH!")


Đã lưu train: (21000, 29) -> train_features.csv
Đã lưu valid: (4500, 29) -> valid_features.csv
Đã lưu test: (4500, 29) -> test_features.csv

Đã lưu cấu hình: feature_config.json
Đã lưu 3 tập dữ liệu phục vụ đánh giá công bằng.

===== KIỂM TRA FILE ĐÃ LƯU =====
Kích thước Train: (21000, 29)
Số giá trị thiếu: 0
Tỷ lệ default: 22.12 %

CELL 5 HOÀN THÀNH!
